# 📧 Mailing Automático de Reporte de Variables Macroeconómicas Argentinas 💵

## 📦 Creación de .env

El siguiente código crea el ambiente en donde se guardarán las cosas en forma secreta, reemplaza con tu contraseña y tus contactos. Luego de creado el ambiente, podés eliminar el chunk.
Para que funcione la contraseña de Gmail, tenés que tener una 'Contraseña de Aplicación', si no tenés, se crea en: https://myaccount.google.com/apppasswords.

IMPORTANTÍSIMO: NO SE USA LA CONTRASEÑA PROPIA CON LA QUE UNO INGRESA AL GMAIL.

## 📚 Instalación de librerías

Para instalar correctamente los paquetes necesarios, puedes utilizar el siguiente código quitando los numerales si es notebook:

In [ ]:
# import sys
#!conda install --yes --prefix {sys.prefix} pandas
#!conda install --yes --prefix {sys.prefix} numpy
#!conda install --yes --prefix {sys.prefix} matplotlib
#!conda install --yes --prefix {sys.prefix} seaborn
#!conda install --yes --prefix {sys.prefix} selenium
#!conda install --yes --prefix {sys.prefix} sqlalchemy
#!conda install --yes --prefix {sys.prefix} webdriver-manager
#!conda install --yes --prefix {sys.prefix} psycopg2-binary

Para la instalación en Visual Studio Code primero vas a necesitar crear un ambiente, luego activarlo y luego instalar allí las librerías necesarias mediante terminal: 

Creación del entorno:
python -m venv venv

Para la activación del entorno:
.\venv\Scripts\activate

Instalación de librerías:
pip install pandas numpy httpx python-dotenv tabulate sqlalchemy psycopg2-binary selenium matplotlib seaborn

Luego yo utilicé para congelar las librerías y crear el archivo de texto que indica qué se requiere instalar para que todo funcione:
pip freeze > requirements.txt

Por otra parte, también vas a necesitar el chromedriver, o el driver necesario para tu navegador web. Este Software se descarga simplemente de la web oficial, https://chromedriver.chromium.org/. Una vez descargado, se ubica en la misma ruta que el script como es en el caso del presente notebook, o dentro del script puede informar la ruta de la siguiente manera, obviamente, quitándo los numerales que convierten las líneas de código en comentarios:

In [ ]:
# ruta_webdriver = r"C:\ruta\webdriver"
# driver = webdriver.Chrome(ruta_webdriver, options=chrome_options)

## 📚 Importación de librerías

In [ ]:
# Estándar de Python
import subprocess
import os
import sys
import time
import json
import locale
import smtplib
import ssl
import datetime as dt
# # No uso getpass ya que es para escribir la contraseña pero es una excelente opción para cuando se quiere tener mayor control humano
#from getpass import getpass

# Manejo de Email
import threading # envio en paralelo
from email.mime.image import MIMEImage
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

# Templates (html/css)
from jinja2 import Environment, FileSystemLoader

# Datos y APIs
import pandas as pd
import numpy as np
import httpx
from dotenv import load_dotenv
from tabulate import tabulate
import yfinance as yf # BTC

# Validación
from models import FilaMacro
from pydantic import ValidationError

# Bases de datos
from sqlalchemy import create_engine
from sqlalchemy import text

# Webscraping (Playwright)
import scrapers
from scrapers.utils import ScraperError

# Visualización
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
import matplotlib.patheffects as pe
from matplotlib.ticker import FuncFormatter, MultipleLocator, AutoMinorLocator
import seaborn as sns

# IA Paragraph
import ia_generator

# Alertas por mail
import mailer


## ⚙️ Configuración: entorno y credenciales

In [ ]:
# Iniciamos cronómetro inmediatamente después de cargar librerías
comienzo = time.perf_counter()

# Carga de variables de entorno
load_dotenv(override=True)  

# Procesamiento de listas de destinatarios
def parse_env_email_list(env_var):
    raw = os.getenv(env_var, "")
    return [email.strip() for email in raw.split(",") if email.strip()]

# Credenciales y Rutas (Uso de mayúsculas para constantes de configuración)
EMAIL_RECEIVER = parse_env_email_list("EMAIL_RECEIVER")
EMAIL_RECEIVER_CSV = parse_env_email_list("EMAIL_RECEIVER_CSV")
EMAIL_SENDER = os.getenv("EMAIL_SENDER")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
RUTA_BBDD = os.getenv("RUTA_BBDD")
RUTA_REPO = os.getenv("RUTA_REPO")
FED_API_KEY = os.getenv("FED_API_KEY")
GEMINI_API_KEY_1 = os.getenv("GEMINI_API_KEY_1")
GEMINI_API_KEY_2 = os.getenv("GEMINI_API_KEY_2")
SUPABASE_DB_URL = os.getenv("SUPABASE_DB_URL")

## 🛠️ Configuración: herramientas

In [ ]:
# Configuración Engine SQLAlchemy para Supabase
engine = create_engine(
    SUPABASE_DB_URL,
    pool_size=3,
    max_overflow=0,
    pool_recycle=300,
    pool_pre_ping=True,
    connect_args={"connect_timeout": 30}
)

# Configuración de Estilo Visual
sns.set(style='ticks')

In [ ]:
# Función para eliminar el signo peso
def eliminar_peso(x):
    x = x.lstrip('$')
    return float(x)

# Función para reemplazar la coma de un string y convertirlo a número
def reemplazar_coma(x):
    x = x.replace(',', '.')
    return float(x)

In [ ]:
# Fuentes para citar en el mail. Las URLs viven en cada módulo: acá solo se leen,
# así no quedan dos copias que se puedan desincronizar.
web_dolarhoy = scrapers.dolarhoy.WEB_DOLARHOY
web_bna = scrapers.bna.WEB_BNA
web_mep = scrapers.ambito.WEB_MEP
web_euro = scrapers.ambito.WEB_EURO
api_riesgo_pais = scrapers.riesgo_pais.API_BASE
bcra_api_url = scrapers.bcra.API_BASE
fed_api_url = scrapers.fed.API_URL

## 📝 Importación de BBDD acumulativa

In [ ]:
# Toma de datos desde Supabase
query_historico = text('SELECT * FROM "Fact_Mercado_Macro" ORDER BY "Fecha" DESC;')
try:
    with engine.connect() as conn:
        df = pd.read_sql_query(query_historico, conn)
    if 'Fecha' in df.columns:
        df['Fecha'] = pd.to_datetime(df['Fecha'], errors='coerce').dt.date.astype(str)
        df = df.dropna(subset=['Fecha'])
    else:
        raise KeyError("❌ Error: La columna 'Fecha' no existe en la tabla de Supabase.")

except Exception as e:
    print(f"❌ Error crítico al leer desde Supabase: {e}")
    print("⚠️ Plan B: Activando contingencia, leyendo desde el CSV de seguridad local...")
    df = pd.read_csv(RUTA_BBDD, encoding='latin1')
    if 'Fecha' in df.columns:
        df['Fecha'] = pd.to_datetime(df['Fecha'], errors='coerce').dt.date.astype(str)
df.head()

In [ ]:
# [# Inserción manual de datos directamente en Supabase
# # Usar cuando el bot no corrió y hay que cargar datos a mano.
# # Para múltiples días: agregar más filas al diccionario (listas de más de 1 elemento).

# new_data = {
#     "Fecha": ["2026-04-06"],    # yyyy-mm-dd
#     "Solidario": [1839.5],
#     "TCC_Blue": [1385.0],
#     "TCV_Blue": [1405.0],
#     "TCV_MEP": [1429.57],
#     "TCC_Billete": [1365.0],
#     "TCV_Billete": [1415.0],
#     "TCC_Divisas": [1383.0],
#     "TCV_Divisas": [1392.5],
#     "TCC_Euro": [1669.75],
#     "TCV_Euro": [1735.75],
#     "fed_tea": [3.64],
#     "bcra_tea": [44.89],
#     "riesgo_pais": [611.0]
# }

# new_data_df = pd.DataFrame(new_data)
# new_data_df['Fecha'] = pd.to_datetime(new_data_df['Fecha'], errors='coerce').dt.strftime('%Y-%m-%d')

# try:
#     with engine.begin() as conn:
#         query_fechas = text('SELECT "Fecha" FROM "Fact_Mercado_Macro"')
#         fechas_db = pd.read_sql(query_fechas, con=conn)
#         fechas_db_set = set(pd.to_datetime(fechas_db['Fecha']).dt.strftime('%Y-%m-%d').tolist()) if not fechas_db.empty else set()

#         df_nuevos = new_data_df.dropna(subset=['Fecha'])
#         df_nuevos = df_nuevos[~df_nuevos['Fecha'].isin(fechas_db_set)]

#         columnas_validas = [c for c in columnas_a_exportar if c in df_nuevos.columns]
#         df_nuevos = df_nuevos[columnas_validas]

#         if not df_nuevos.empty:
#             print(f"⏳ Supabase: Procesando {len(df_nuevos)} registros potenciales...")
#             insertados = 0
#             for idx, fila in df_nuevos.iterrows():
#                 try:
#                     pd.DataFrame([fila]).to_sql(
#                         name='Fact_Mercado_Macro',
#                         con=conn,
#                         if_exists='append',
#                         index=False
#                     )
#                     insertados += 1
#                 except Exception as row_err:
#                     print(f"⚠️ No se pudo insertar la fecha {fila.get('Fecha', 'Desconocida')}. Motivo: {row_err}")
#         else:
#             print("ℹ️ Supabase: Los datos de esas fechas ya existen. No se insertó nada.")
#         print(f"🚀 Finalizó. Se insertaron {insertados} filas.")
# except Exception as e:
#     print(f"❌ Error crítico al conectar con Supabase: {e}")
# ]

## Consumo de APIs

In [ ]:
# Todas las fuentes en paralelo: 3 webs con Playwright y 3 APIs con httpx.
# Si alguna falla, avisa por mail ANTES de cortar: sin esto la corrida moría en silencio.
try:
    bna_result, dolarhoy_result, ambito_result, riesgo_pais_result, bcra_result, fed_result = scrapers.run_all_sync()
except ScraperError as e:
    mailer.alertar_scraper_caido(e)
    raise

In [ ]:
# BADLAR efectiva anual (variable 140) e inflación mensual (27), ya traídas por
# scrapers.bcra. La serie llega ascendente, que es el orden que necesita el rolling.
bcra_tea = bcra_result["bcra_tea"]

if bcra_result["bcra_tea_fecha"] != pd.Timestamp.today().date():
    print(f"⚠️ BCRA: la última TEA publicada es del {bcra_result['bcra_tea_fecha']}, no de hoy.")

inflacion_df = pd.DataFrame(
    bcra_result["inflacion_mensual"], columns=["Fecha", "Inflación Mensual"]
)
inflacion_df["Fecha"] = pd.to_datetime(inflacion_df["Fecha"])

# Cálculos acumulados

factor = inflacion_df["Inflación Mensual"] / 100 + 1

inflacion_df["Inflación Bimestral"] = (
    factor.rolling(2).apply(np.prod, raw=True) - 1
) * 100

inflacion_df["Inflación Trimestral"] = (
    factor.rolling(3).apply(np.prod, raw=True) - 1
) * 100

inflacion_df["Inflación Anual"] = (
    factor.rolling(12, min_periods=12).apply(np.prod, raw=True) - 1
) * 100

inflacion_df["bcra_tea"] = bcra_tea

# Output

inflacion_df = (
    inflacion_df
    .iloc[::-1]
    .head(12)
)

inflacion_df["Fecha"] = inflacion_df["Fecha"].dt.strftime("%d-%m-%Y")

# Separo el valor de la tasa del BCRA en un dataframe aparte
bcra_tea_ultimo = inflacion_df["bcra_tea"].iloc[0]
bcra_tea_df = pd.DataFrame(
    {"bcra_tea": [bcra_tea_ultimo]}
)

display(
    inflacion_df.style.format({
        "Inflación Mensual": "{:.2f}%",
        "Inflación Bimestral": "{:.2f}%",
        "Inflación Trimestral": "{:.2f}%",
        "Inflación Anual": "{:.2f}%",
        "bcra_tea": "{:.2f}%"
    })
)

In [ ]:
# EFFR de la API de St. Louis FED, ya traída por scrapers.fed
fed_tea = pd.DataFrame(
    {"fed_tea": [fed_result["fed_tea"]]},
    index=[0]
)

fed_tea

## 🤖 Ingreso a la web y toma de datos 

In [ ]:
# Empezamos a convertir los resultados de los scrapers en DataFrames
dolar_hoy = pd.DataFrame(
    {"Fecha": dt.datetime.today(),
     "TCC_Blue": dolarhoy_result["TCC_Blue"],
     "TCV_Blue": dolarhoy_result["TCV_Blue"]
    },
    index=[0]
    )

dolar_hoy["Fecha"] = pd.to_datetime(dolar_hoy["Fecha"]).dt.date
dolar_hoy

In [ ]:
bna = pd.DataFrame([bna_result])
bna

In [ ]:
euro = pd.DataFrame(
    {"TCC_Euro": [ambito_result["TCC_Euro"]],
    "TCV_Euro": [ambito_result["TCV_Euro"]]},
    index=[0]
)
euro

In [ ]:
ambito = pd.DataFrame(
    {"TCV_MEP": [ambito_result["TCV_MEP"]]},
    index=[0])
ambito

In [ ]:
riesgo_pais = pd.DataFrame(
    {"riesgo_pais": [riesgo_pais_result["riesgo_pais"]]},
    index=[0])

# La API publica el riesgo país con un día hábil de rezago. Avisamos cuando el valor
# no corresponde a hoy, para no repetir el desfase que tenía el scraper de Ambito.
_fecha_rp = riesgo_pais_result["riesgo_pais_fecha"]
if _fecha_rp != pd.Timestamp.today().date():
    print(f"⚠️ Riesgo país: el último dato publicado es del {_fecha_rp}, no de hoy.")

ambito = pd.concat([ambito, riesgo_pais], axis=1)
ambito

In [ ]:
# Nueva fila final con todos los datos tomados
fila_nueva = pd.concat([dolar_hoy, bna, ambito, euro, fed_tea, bcra_tea_df], axis=1)
fila_nueva

## 🖥️ Concatenado de la nueva fila en el DataFrame acumulado 

In [ ]:
# Si la prueba con Pydantic da error en la fila nueva, manda un mail al dev para que haga un arreglo
def enviar_mail_error(mensaje_error: str):
    """Mail de texto plano a EMAIL_RECEIVER_CSV cuando la validación falla."""
    em = MIMEMultipart()
    em["From"] = EMAIL_SENDER
    em["To"] = ", ".join(EMAIL_RECEIVER_CSV)
    em["Subject"] = "Atención: Error en el mailing automático"
    em.attach(MIMEText(str(mensaje_error), "plain"))
    context = ssl.create_default_context()
    with smtplib.SMTP("smtp.gmail.com", 587) as smtp:
        smtp.ehlo()
        smtp.starttls(context=context)
        smtp.login(EMAIL_SENDER, EMAIL_PASSWORD)
        smtp.sendmail(EMAIL_SENDER, EMAIL_RECEIVER_CSV, em.as_string())
    print("📧 Mail de error enviado.")

try:
    FilaMacro(**fila_nueva.iloc[0].to_dict())
    print("✅ Validación Pydantic OK — todos los campos correctos.")
except ValidationError as e:
    enviar_mail_error(e)
    raise

In [ ]:
# Concatenado de la fila nueva
df = pd.concat([
    fila_nueva,
    df
]).reset_index(drop=True)

In [ ]:
# Paridad de tasas de interés de Irving Fisher a 3 meses
bcra_tea_numerador = 1 + (float(df["bcra_tea"].iloc[0])/100)
fed_tea_denominador = 1 + (float(df["fed_tea"].iloc[0])/100)
division = bcra_tea_numerador / fed_tea_denominador
fwd_oficial = float(df["TCV_Billete"].iloc[0]) * division
fwd_blue = float(df["TCV_Blue"].iloc[0]) * division

In [ ]:
# Crear columnas de cálculo de brechas y variación del día
df["Solidario / TCV Blue"] = df["Solidario"] / df["TCV_Blue"] - 1
df["TCV MEP / TCV Blue"] = df["TCV_MEP"] / df["TCV_Blue"] - 1
df["TCV Euro / TCC Blue %"] = df["TCV_Euro"] / df["TCV_Blue"] - 1

# Forward fill para manejar los porcentajes vacíos
df["Solidario"] = df["Solidario"].ffill()
df["TCV_Blue"] = df["TCV_Blue"].ffill()
df["TCV_Euro"] = df["TCV_Euro"].ffill()

# Calcular la variación del día
df["Variación Solidario"] = df["Solidario"].pct_change(periods=-1).fillna(0)
df["Variación TCV Blue"] = df["TCV_Blue"].pct_change(periods=-1).fillna(0)
df["Variación TCV Euro"] = df["TCV_Euro"].pct_change(periods=-1).fillna(0)

df.head()

## 💾 Exportar a CSV y Actualizar fila en SQL

In [ ]:
# Estructura exacta de la tabla "Fact_Mercado_Macro" en Supabase
columnas_a_exportar = [
    "Fecha",    
    "TCC_Blue",     
    "TCV_Blue",     
    "TCC_Billete",  
    "TCV_Billete",  
    "TCC_Divisas",     
    "TCV_Divisas",     
    "Solidario",      
    "TCV_MEP",   
    "riesgo_pais",
    "TCC_Euro",
    "TCV_Euro",
    "fed_tea",
    "bcra_tea",  
    "ai_paragraph",
    "ai_model"
]

try:
    with engine.begin() as conn:
        
        # Traer fechas existentes para evitar duplicados en la PK
        query_fechas = text('SELECT "Fecha" FROM "Fact_Mercado_Macro"')
        fechas_db = pd.read_sql(query_fechas, con=conn)
        
        if not fechas_db.empty:
            fechas_db_set = set(pd.to_datetime(fechas_db['Fecha']).dt.strftime('%Y-%m-%d').tolist())
        else:
            fechas_db_set = set()

        # Aislamiento del DataFrame y Control de Daños
        df_load = df.copy()
        df_load['Fecha'] = pd.to_datetime(df_load['Fecha'], format='mixed', errors='coerce').dt.strftime('%Y-%m-%d')
        
        # Filtrar registros inexistentes
        df_load = df_load.dropna(subset=['Fecha'])
        df_nuevos = df_load[~df_load['Fecha'].isin(fechas_db_set)]

        # Asegurar concordancia de esquema exacto antes de insertar
        columnas_validas = [c for c in columnas_a_exportar if c in df_nuevos.columns]
        df_nuevos = df_nuevos[columnas_validas]

        # Carga defensiva fila por fila
        if not df_nuevos.empty:
            print(f"⏳ Supabase: Procesando {len(df_nuevos)} registros potenciales...")
            insertados = 0
            
            for idx, fila in df_nuevos.iterrows():
                try:
                    df_fila = pd.DataFrame([fila])
                    df_fila.to_sql(
                        name='Fact_Mercado_Macro',
                        con=conn,
                        if_exists='append',
                        index=False
                    )
                    insertados += 1
                except Exception as row_err:
                    print(f"⚠️ No se pudo insertar la fecha {fila.get('Fecha', 'Desconocida')}. Motivo: {row_err}")
                    continue
            
            print(f"🚀 Carga finalizada. Se insertaron exitosamente {insertados} filas de {len(df_nuevos)} evaluadas.")
        else:
            print("ℹ️ Supabase: El pipeline detectó que los datos ya están al día. No se requirieron inserciones.")

except Exception as e:
    print(f"❌ Error crítico en la conexión o lectura de Supabase: {e}")

finally:
    # Bloque único y seguro de liberación de recursos
    if 'engine' in locals():
        engine.dispose()
        print("🔌 Pool de conexiones de SQLAlchemy liberado correctamente.")

In [ ]:
parrafo_ia = ia_generator.procesar_y_guardar_parrafo(engine)
print(parrafo_ia)

## 📈 Visualizaciones

In [ ]:
# Tomamos las primeras filas que querramos mostrar y las primeras 14 columnas, es decir, sin variaciones
cotizaciones_a_mostrar = 25
# Copia del dataframe, para no reemplazarlo
data = df.copy()
# Damos vuelta el dataframe sólo para los gráficos
# Si no lo damos vuelta, los gráficos van a empzar de hoy hacia atrás
data = data.iloc[::-1]
data['Fecha'] = pd.to_datetime(data['Fecha'], dayfirst=False, errors='coerce')
data['Fecha'] = data['Fecha'].dt.strftime('%d/%m/%y')
data.tail()

In [ ]:
# Creo un subplot de 3 gráficos verticales
fig, ax = plt.subplots(
    3, 1,
    figsize=(10, 14),
    sharex=False
)

# Estilo de gráfico, con ticks en X e Y
sns.set(style="ticks")

titulos = [
    "Cotizaciones Paralelas",
    "Cotizaciones Oficiales",
    "Evolución del Riesgo País",
]

ax0_palette = sns.color_palette("dark")

ax0_columns = [
    "TCV_Euro",
    "TCV_Blue",
    "TCC_Blue",
    "TCV_MEP",
    "Solidario"
]
ax0_labels = [
    "TCV Euro Blue",
    "TCV Blue",
    "TCC Blue",
    "MEP",
    "Solidario"
]

# Títulos general y para cada gráfico
fig.suptitle(
    f"Tipos de cambio y riesgo país - últimas {cotizaciones_a_mostrar} cotizaciones",
    fontweight="bold",
    fontsize=18
)
for i, titulo in enumerate(titulos):
    ax[i].set_title(
        titulo,
        fontweight="bold",
        fontsize=12
    )

for i, column in enumerate(ax0_columns):
    sns.lineplot(
        x="Fecha",
        y=column,
        data=data.tail(cotizaciones_a_mostrar),
        label=ax0_labels[i],
        color=ax0_palette[i],
        marker="o",
        ax=ax[0],
    )
    
# Etiquetas de datos para cada 5 valores y último valor en el primer gráfico
for i, (column, label) in enumerate(zip(ax0_columns, ax0_labels)):
    for j, (x, y) in enumerate(zip(data["Fecha"].tail(cotizaciones_a_mostrar), data[column].tail(cotizaciones_a_mostrar))):
        if j % 5 == 0 or j == len(data[column].tail(cotizaciones_a_mostrar)) - 1:  # Agregar etiqueta de datos cada 5 valores
            ax[0].annotate(
                f"{y:,.2f}",
                xy=(x, y),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontproperties=fm.FontProperties(weight="bold", size=9),
                bbox=dict(boxstyle="round", edgecolor=ax0_palette[i], facecolor="white")
            )
            
# Gráfico de cotizaciones oficiales
ax1_palette = sns.color_palette("bright")
ax1_columns = [
    "TCV_Billete",
    "TCV_Divisas",
    "TCC_Divisas",
    "TCC_Billete"
]
ax1_labels = [
    "TCV Billete",
    "TCV Divisas",
    "TCC Divisas",
    "TCC Billete"
]
for i, column in enumerate(ax1_columns):
    sns.lineplot(
        x="Fecha",
        y=column,
        data=data.tail(cotizaciones_a_mostrar),
        label=ax1_labels[i],
        color=ax1_palette[i],
        marker="D",
        ax=ax[1],
    )

# Etiquetas de datos para cada 5 valores y último valor en el segundo gráfico
for i, (column, label) in enumerate(zip(ax1_columns, ax1_labels)):
    for j, (x, y) in enumerate(zip(data["Fecha"].tail(cotizaciones_a_mostrar), data[column].tail(cotizaciones_a_mostrar))):
        if j % 5 == 0 or j == len(data[column].tail(cotizaciones_a_mostrar)) - 1:  # Agregar etiqueta de datos cada 5 valores
            ax[1].annotate(
                f"{y:,.2f}",
                xy=(x, y),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontproperties=fm.FontProperties(weight="bold", size=9),
                bbox=dict(boxstyle="round", alpha=0.4, edgecolor=ax1_palette[i], facecolor="white")
            )

# Preparamos los datos y las categorías de color
df_riesgo = data.tail(cotizaciones_a_mostrar).copy()
max_riesgo = df_riesgo['riesgo_pais'].max()
min_riesgo = df_riesgo['riesgo_pais'].min()

def categorizar_riesgo(valor):
    if abs(valor - max_riesgo) < 1e-2: return 'Máximo'
    if abs(valor - min_riesgo) < 1e-2: return 'Mínimo'
    return 'Normal'

df_riesgo['Categoria'] = df_riesgo['riesgo_pais'].apply(categorizar_riesgo)

# Variacion respecto al dia anterior. Se calcula sobre la serie COMPLETA y recien
# despues se recorta: asi la primera barra compara contra su dia previo real en
# vez de quedarse sin dato.
df_riesgo['var_pct'] = (data['riesgo_pais'].pct_change() * 100).tail(cotizaciones_a_mostrar).values

# Colores de la variacion. Rojo = sube el riesgo (malo), verde = baja.
# Medidos sobre blanco: 5.44:1 y 4.72:1, los dos pasan WCAG AA para texto chico.
# El verde del mail (#27ae60) daba 2.87:1 y se descarto por ilegible.
VAR_SUBE = "#c0392b"
VAR_BAJA = "#1e8449"
VAR_IGUAL = "#000000"

# Gráfico de barras usando HUE y el mapeo explícito
sns.barplot(
    x="Fecha",
    y="riesgo_pais",
    data=df_riesgo,
    hue="Categoria", # Esto le dice a Seaborn de dónde sacar el grupo
    palette={'Máximo': 'red', 'Mínimo': 'green', 'Normal': 'silver'}, # Colores fijos por grupo
    legend=False, # Evitamos que aparezca el cuadrito de leyenda
    width=0.8,
    edgecolor="black",
    linewidth=2,
    ax=ax[2],
)

# Aplicar borde punteado SOLO al máximo y mínimo
for bar in ax[2].patches:
    height = bar.get_height()
    if height in [max_riesgo, min_riesgo]:
        bar.set_linestyle((0, (6, 2, 2, 3)))  # punteado para max y min
    else:
        bar.set_linestyle("solid")             # sólido para el resto


# Agregamos etiquetas de datos a las barras, tercer gráfico
valores_riesgo = df_riesgo["riesgo_pais"].tolist()
variaciones_riesgo = df_riesgo["var_pct"].tolist()

for bar in ax[2].patches:
    height = bar.get_height()
    if not height:
        continue

    # La posicion en X identifica la barra. No se usa el indice del enumerate
    # porque seaborn, al agrupar por hue, no garantiza que patches venga en el
    # mismo orden que las filas del dataframe.
    centro = bar.get_x() + bar.get_width() / 2
    idx = int(round(centro))
    if not (0 <= idx < len(valores_riesgo)):
        continue

    # Aplicar efectos de trazo si es el valor máximo o mínimo
    path_effects = [pe.withStroke(linewidth=1.5, foreground="black")] if height in [max_riesgo, min_riesgo] else []

    # Ajustar el tamaño de la fuente dependiendo si es el valor máximo o mínimo
    font_size = 12 if height in [max_riesgo, min_riesgo] else 9

    # Cambiar color de la anotación dependiendo si es máximo o mínimo
    annotation_color = 'red' if height == max_riesgo else 'green' if height == min_riesgo else 'black'

    # Configurar las propiedades del recuadro alrededor de las anotaciones
    bbox_props = dict(boxstyle="round", alpha=0.4, edgecolor=annotation_color, facecolor="white", path_effects=path_effects)

    # Valor del riesgo pais, arriba de todo
    ax[2].annotate(
        f"{height:,.0f}",
        xy=(centro, height + 2.25),
        xytext=(0, 22),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontproperties=fm.FontProperties(weight="bold", size=font_size),
        bbox=bbox_props,
        color=annotation_color,
        path_effects=path_effects
    )

    # Variacion diaria, justo debajo del valor.
    # La flecha no es decorativa: repite la informacion del color, asi el dato
    # sigue siendo legible para alguien que no distingue rojo de verde.
    # Se usa un caracter y no un emoji porque matplotlib no renderiza emoji a
    # color: saldria un cuadrito vacio.
    variacion = variaciones_riesgo[idx]
    if pd.isna(variacion):
        continue
    if round(variacion, 1) == 0:
        etiqueta_var, color_var = "0%", VAR_IGUAL
    elif variacion > 0:
        etiqueta_var, color_var = f"\u25b2{variacion:.1f}%", VAR_SUBE
    else:
        etiqueta_var, color_var = f"\u25bc{abs(variacion):.1f}%", VAR_BAJA

    ax[2].annotate(
        etiqueta_var,
        xy=(centro, height),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontproperties=fm.FontProperties(weight="bold", size=7.5),
        color=color_var,
    )

for axis in ax:
    axis.set_xlabel("")
    axis.xaxis.set_major_locator(MultipleLocator(2))
    axis.xaxis.set_minor_locator(MultipleLocator(1))

# y separador de miles, por si la inflación se va demasiado, ya estoy adelantado
for i in range(2):
    ax[i].yaxis.set_major_formatter(FuncFormatter("{:,.2f}".format))

# Títulos y ticks eje Y
y_labels = ["ARS / USD", "ARS / USD", "Puntos Base"]
y_major_locators = [
    MultipleLocator(50),
    MultipleLocator(10),
    MultipleLocator(500)
]
y_minor_locators = [
    MultipleLocator(250),
    MultipleLocator(25),
    MultipleLocator(200)
]
for i in range(3):
    ax[i].set_ylabel(
        y_labels[i],
        fontsize = 12,
        fontweight="bold"
    )
    ax[i].yaxis.set_major_locator(y_major_locators[i])
    ax[i].yaxis.set_minor_locator(y_minor_locators[i])

# Límites eje Y
# 1.22 y no 1.1: ahora hay dos renglones de etiqueta sobre cada barra
ax[2].set_ylim([0, data.tail(cotizaciones_a_mostrar).riesgo_pais.max() * 1.22])

# Le damos estilo al grid del fondo
grid_estilo = {"color": "silver", "linestyle": "--", "linewidth": 0.5}
for axis in ax:
    axis.grid(**grid_estilo)
    
ax[0].legend(prop={'size': 8}, loc="upper left", shadow=True)
ax[1].legend(prop={'size': 8}, loc="upper left",shadow=True)
   
# Guardamos el gráfico como imagen .jpg para enviarla por mail
fig.tight_layout(pad=1)
graficos_jpg = "Gráficos Tipos de Cambios y Riesgo País.jpg"
plt.savefig("Previews/" + graficos_jpg)

In [ ]:
#locale.setlocale(locale.LC_TIME, 'es_ES.utf8')
locale.setlocale(locale.LC_TIME, 'en_US.utf8')

# Para inflacion_df (que sí es todo dd-mm-YYYY)
inflacion_df["Fecha"] = pd.to_datetime(inflacion_df["Fecha"], format="%d-%m-%Y")

# Para data (que tiene formatos mezclados)
data["Fecha"] = pd.to_datetime(data["Fecha"], dayfirst=True, errors="coerce")

variacion_acumulada = data.merge(inflacion_df[["Fecha","Inflación Mensual"]], on=["Fecha"], how="outer")

#Ordenamos por fecha ya que luego del merge, pone al final las fechas sin cotización pero con inflación
variacion_acumulada.sort_values("Fecha", inplace=True)

# # Filtramos por el año actual
# current_year = pd.Timestamp.today().year
fecha_inicio = pd.to_datetime("2025-07-01")
variacion_acumulada = variacion_acumulada[variacion_acumulada["Fecha"] >= fecha_inicio]

#Creamos las columnas acumulativas
columns_to_cumulate = ['Variación Solidario', 'Variación TCV Blue', 'Inflación Mensual']
for column in columns_to_cumulate:
    if column == 'Inflación Mensual':
        variacion_acumulada[f'{column} Acumulada'] = ((1 + variacion_acumulada[column] / 100).cumprod()-1)*100
    else:
        variacion_acumulada[f'{column} Acumulada'] = ((1 + variacion_acumulada[column]).cumprod()-1)*100
        
fig, ax = plt.subplots(figsize=(10, 5))

# Gráfico de cotizaciones oficiales
palette = sns.color_palette("pastel")
ax2_columns = [
    "Variación Solidario Acumulada",
    "Variación TCV Blue Acumulada",
    "Inflación Mensual Acumulada"
]
ax2_labels = [
    "Δ Solidario Acumulada",
    "Δ TCV Blue Acumulada",
    "Inflación Mensual Acumulada"
]

for i, column in enumerate(ax2_columns):

    if column == "Inflación Mensual Acumulada":
        sns.lineplot(
            x="Fecha",
            y=column,
            data=variacion_acumulada,
            label=ax2_labels[i],
            color=palette[i],
            linewidth=2,
            marker="D",
            markersize=7,
            drawstyle='steps-pre'
        )
    else:
        sns.lineplot(
            x="Fecha",
            y=column,
            data=variacion_acumulada,
            label=ax2_columns[i],
            color=palette[i],
            linewidth=2
        )

    # Etiqueta de datos para cada línea, se requieren diferentes frecuencias
    bbox = dict(facecolor=palette[i], edgecolor=palette[i], boxstyle='square,pad=0.2')
    label_frequency = 25 if column != "Inflación Mensual Acumulada" else 1
    for idx, row in variacion_acumulada.iterrows():
        if not pd.isna(row[column]):
            # Etiquetar según frecuencia correspondiente
            if idx % label_frequency == 0:
                plt.text(
                    row['Fecha'],
                    row[column]*1.05,
                    f'{row[column]:,.1f}',
                    fontsize=9,
                    color="black",
                    ha="center",
                    va="center",
                    bbox=bbox
                )

            # Etiqueta del valor máximo por línea/columna
            bbox_max = dict(facecolor=palette[i], edgecolor="black", boxstyle='square,pad=0.2')
            if row[column] == variacion_acumulada[column].max() and idx == variacion_acumulada[column].idxmax():
                plt.text(
                    row['Fecha'],
                    row[column]*1.05,
                    f'{row[column]:,.2f}',
                    fontsize=10,
                    color="darkred",
                    fontweight="bold",
                    ha="center",
                    va="center",
                    bbox=bbox_max
                )

ax.set_title("Variaciones acumuladas", fontsize=14, fontweight="bold")

# Para que los meses aparezcan en español
locale.setlocale(locale.LC_TIME, 'es_ES.utf8')
# Formatear eje de X
ax.set_xticklabels(variacion_acumulada.Fecha[::3], rotation=45)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))  # Mostrar un tick al mes
ax.set_xlabel("")

# Formatear eje de Y
ax.yaxis.set_major_formatter(FuncFormatter("{:,.1f} %".format))
ax.set_xlabel("")
ax.set_ylabel("Variación Acumulada", fontsize = 12, fontweight="bold")

plt.grid(True)
sns.despine()

variaciones_jpg = "Variaciones.jpg"
plt.savefig("Previews/" + variaciones_jpg)

plt.legend(prop={'size': 8}, shadow=True)
plt.show()

In [ ]:
# Para que los meses aparezcan en español
locale.setlocale(locale.LC_TIME, 'es_ES.utf8')

# Crear un subplot con tres gráficos verticales
fig, ax = plt.subplots(figsize=(10, 7), sharex=True)
ax2 = plt.twinx() #comparten eje X con ejes Ys diferentes

inflacion_df=inflacion_df[::-1]
inflacion_df["Fecha"] = pd.to_datetime(inflacion_df["Fecha"], dayfirst=True).dt.strftime("%b/%Y").str.replace('.', '')
largo_fecha = len(inflacion_df["Fecha"])

fig.suptitle(
    f"Evolución de la Inflación Argentina en los últimos {largo_fecha} meses",
    fontweight="bold",
    fontsize=18
)

# Gráfico inflación mensual NO ACUMULADA
inflacion_mensual_barplot = sns.barplot(
    x="Fecha",
    y="Inflación Mensual",
    data=inflacion_df,
    edgecolor="black",
    facecolor="darkgrey",
    label="Inflación Mensual",
    ax=ax,
    legend=False
)

# Gráfico inflación mensual ACUMULADA
inflacion_acumulada_lineplot = sns.lineplot(
    x="Fecha",
    y="Inflación Anual",
    data=inflacion_df,
    linewidth=2.5,
    color="darkred",
    marker='o',
    markersize=8,
    label="Inflación Anual",
    ax=ax2
)

# Agregar etiquetas de datos para inflación mensual
bbox = dict(boxstyle="round,pad=0.3", edgecolor="darkred", facecolor="white")
path_effects=[pe.withStroke(linewidth=0.5, foreground="black")]
max_y = inflacion_df["Inflación Mensual"].max()
max_x = inflacion_df["Fecha"][inflacion_df["Inflación Mensual"].idxmax()]

for x, y in zip(inflacion_df["Fecha"], inflacion_df["Inflación Mensual"]):
    if y != max_y:
        ax.annotate(
            f'{y:.1f}%', 
            (x, y), 
            textcoords="offset points", 
            xytext=(0, 12), 
            ha='center',
            bbox=bbox
        )

# Resaltar el valor máximo de inflación mensual
ax.annotate(
    f'{max_y:.1f}%', 
    (max_x, max_y), 
    textcoords="offset points", 
    xytext=(0, 12), 
    ha='center',
    fontsize=12,
    fontweight='bold',
    color='darkred',
    bbox=bbox,
    path_effects=path_effects
)
   
# Agregar etiquetas de datos para inflación acumulada
max_y = inflacion_df["Inflación Anual"].max()
max_x = inflacion_df["Fecha"][inflacion_df["Inflación Anual"].idxmax()]

# Resaltar el valor máximo de inflación acumulada
ax2.annotate(
    f'{max_y:.1f}%', 
    (max_x, max_y), 
    textcoords="offset points", 
    xytext=(0, 12), 
    ha='center',
    fontsize=12,
    fontweight='bold',
    color='darkred',
    bbox=bbox,
    path_effects=path_effects
)

# Configurar los ticks del eje X para mostrar cada x meses
ax.set_xlabel("")
ax.set_xticks(range(0, largo_fecha, 1))
ax.set_xticklabels(inflacion_df.Fecha[::1], rotation=45)
ax.set_xlim(-0.5, largo_fecha -0.5)

# Configurar los ticks del eje Y
ax.set_ylim([0, inflacion_df["Inflación Mensual"].max() * 1.2])
ax.set_ylabel("Inflación Mensual", fontweight="bold", fontsize=12)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.1f} %"))

# Configurar los ticks del segundo eje Y
ax2.set_ylabel("Inflación Anual", fontweight="bold", fontsize=12, rotation=270, labelpad=15)
ax2.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.0f} %"))

#Leyenda, es más difícil cuando se usa twinsx
handles1, labels1 = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
plt.legend(handles1 + handles2, labels1 + labels2, prop={'size': 8}, shadow=True)

# Mostrar el gráfico y lo guardamos en un .jpg para enviar por correo
inflacion_jpg = "Gráficos Inflación.jpg"
plt.savefig("Previews/" + inflacion_jpg)
plt.show()

In [ ]:
# Configurar estilo con seaborn
sns.set_theme(style='darkgrid', palette='husl')
plt.style.use('dark_background')

# Obtener datos (con margen extra para que las medias móviles y la volatilidad
# ya tengan valores válidos desde el primer día que se muestra en el gráfico)
end_date = dt.datetime.now()
start_date = end_date - dt.timedelta(days=365)
fetch_start_date = start_date - dt.timedelta(days=40)

btc_df = yf.download('BTC-USD', start=fetch_start_date, end=end_date, interval='1d', progress=False)
btc_df = btc_df.reset_index()

# Aplanar MultiIndex si existe
if isinstance(btc_df.columns, pd.MultiIndex):
    btc_df.columns = btc_df.columns.get_level_values(0)

# Detectar columna de fecha y usarla como índice
btc_df['index'] = pd.to_datetime(btc_df['index'])
btc_df = btc_df.set_index('index')
btc_df.index.name = 'Date'

# Calcular medias móviles y volatilidad (usando el margen extra ya descargado)
btc_df['MA7'] = btc_df['Close'].rolling(window=7).mean()
btc_df['MA30'] = btc_df['Close'].rolling(window=30).mean()
btc_df['Volatility'] = btc_df['Close'].pct_change().rolling(window=20).std() * 100

# Recortar al rango de 12 meses a mostrar, ya con las medias/volatilidad completas
# (evita el hueco vacío al inicio del gráfico por los NaN de los rolling windows)
btc_df = btc_df[btc_df.index >= start_date]

# Obtener valores importantes
# .iloc[-1] es la ULTIMA vela disponible, no necesariamente la de hoy: yfinance
# cierra el dia segun UTC y puede venir con retraso. Por eso la etiqueta dice
# 'ultima cotizacion' y no 'precio hoy'.
price_last = float(btc_df['Close'].iloc[-1])
price_last_date = btc_df.index[-1]
price_max = float(btc_df['Close'].max())
price_min = float(btc_df['Close'].min())
volatility_current = float(btc_df['Volatility'].iloc[-1])

# Crear figura con estilo superior
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), facecolor='#0a0a0a',
                                gridspec_kw={'height_ratios': [3, 1]})

for ax in [ax1, ax2]:
    ax.set_facecolor('#0a0a0a')
    ax.grid(True, alpha=0.2, color='#333333', linestyle='-', linewidth=0.5)

# Gráfico principal con área rellena
ax1.fill_between(btc_df.index, btc_df['Close'], alpha=0.1, color='#00ff41')
ax1.plot(btc_df.index, btc_df['Close'], color='#00ff41', linewidth=2.5, label='Precio BTC', zorder=5)

# Medias móviles con mejor estilo
ax1.plot(btc_df.index, btc_df['MA7'], color='#FFD700', linewidth=2.2,
         linestyle='-.', label='Media 7 días', alpha=0.9, zorder=4)
ax1.plot(btc_df.index, btc_df['MA30'], color='#FF6B6B', linewidth=2.2,
         linestyle=':', label='Media 30 días', alpha=0.85, zorder=4)

# Configurar eje X principal
ax1.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.xaxis.set_minor_locator(mdates.MonthLocator(bymonthday=15))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax1.set_xlim(btc_df.index[0], btc_df.index[-1])
ax1.set_ylim(bottom=0)

# Títulos y etiquetas
ax1.set_title('BTC/USD - Últimos 12 Meses con Seaborn Style', fontsize=20, fontweight='bold',
              color='#00ff41', pad=20)
ax1.set_ylabel('USD/BTC', fontsize=13, color='#00ff41', fontweight='bold')

# Leyenda principal
legend1 = ax1.legend(loc='upper left', fontsize=12, framealpha=0.95,
                     edgecolor='#00ff41', facecolor='#0a0a0a', labelcolor='#ffffff',
                     title='Indicadores', title_fontsize=12)
legend1.get_title().set_color('#00ff41')

# Gráfico de volatilidad
ax2.fill_between(btc_df.index, btc_df['Volatility'], alpha=0.3, color='#FF6B6B')
ax2.plot(btc_df.index, btc_df['Volatility'], color='#FF6B6B', linewidth=2, label='Volatilidad (20d)')
ax2.set_ylabel('Volatilidad (%)', fontsize=12, color='#FF6B6B', fontweight='bold')
ax2.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.xaxis.set_minor_locator(mdates.MonthLocator(bymonthday=15))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax2.set_xlim(btc_df.index[0], btc_df.index[-1])
ax2.set_ylim(bottom=0)
ax2.legend(loc='upper left', fontsize=11, framealpha=0.95,
           edgecolor='#FF6B6B', facecolor='#0a0a0a', labelcolor='#ffffff')

# Ejes
for ax in [ax1, ax2]:
    ax.spines['bottom'].set_color('#00ff41')
    ax.spines['left'].set_color('#00ff41')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # Ticks mayores y menores visibles en ambos ejes
    ax.minorticks_on()
    ax.tick_params(axis='both', which='major', direction='in', length=8, width=1.5,
                   color='#00ff41', labelcolor='#ffffff', labelsize=10,
                   bottom=True, left=True)
    ax.tick_params(axis='both', which='minor', direction='in', length=4, width=1.0,
                   color='#00ff41', bottom=True, left=True)

# Etiquetas de información (KPIs)
kpi_text = (f'Última cotización: ${price_last:,.0f}  ({price_last_date:%d/%m/%y})\n'
            f'Máximo: ${price_max:,.0f}\n'
            f'Mínimo: ${price_min:,.0f}\n'
            f'Volatilidad: {volatility_current:.2f}%')
ax1.text(0.98, 0.97, kpi_text, transform=ax1.transAxes, fontsize=11, fontweight='bold',
         bbox=dict(boxstyle='round,pad=1', facecolor='#1a1a1a', edgecolor='#00ff41', alpha=0.9, linewidth=2),
         ha='right', va='top', color='#00ff41', family='monospace')

plt.tight_layout()

btc_jpg = "Gráfico BTC.jpg"
plt.savefig("Previews/" + btc_jpg, dpi=150, facecolor='#0a0a0a', edgecolor='none', bbox_inches='tight')
plt.show()

## 📮 Mailing automático

In [ ]:
# Tomamos sólo los últimos 12 meses
inflacion_df = inflacion_df.tail(12).copy()

# Tomamos el último valor de inflación interanual
inflacion_interanual = inflacion_df["Inflación Anual"].iloc[-1]
inflacion_interanual = "{0:,.2f}%".format(inflacion_interanual)

# Columnas a convertir en % y los dividimos por 100
cols_pct = [
    "Inflación Mensual",
    "Inflación Bimestral",
    "Inflación Trimestral", 
    "Inflación Anual",
    "bcra_tea"
]
inflacion_df[cols_pct] = inflacion_df[cols_pct] / 100
inflacion_df

In [ ]:
# Dividimos las TEA por 100
df['fed_tea'] = df['fed_tea'].apply(lambda x: f"{x/100:.2%}" if pd.notnull(x) else "")
df['bcra_tea'] = df['bcra_tea'].apply(lambda x: f"{x/100:.2%}" if pd.notnull(x) else "")

In [ ]:
# Descomentar la siguiente línea para testear
# EMAIL_RECEIVER = EMAIL_SENDER

# Aseguramos formato de fecha en df principal
df['Fecha'] = pd.to_datetime(df['Fecha']).dt.strftime("%d/%m/%y")

cotizaciones = [
    "Fecha",
    "tccblue",
    "tcvblue",
    "tccBillete",
    "tcvBillete",
    "tccDivisas",
    "tcvDivisas",
    "Solidario",
    "MEP",
    "RiesgoPaís",
    "tccEuros",
    "tcvEuros",
    "FEDtea",
    "BCRAtea"
]

# Valores correspondientes a los últimos días, 14 columnas
valores = df.iloc[:cotizaciones_a_mostrar, :14].values.tolist()
tabla_cotizaciones = tabulate(
    valores,
    headers=cotizaciones,
    tablefmt = "html",
    numalign="center",
    floatfmt=".2f"
)

# Encabezados para tabular últimos 10 días del df y ponerlo en el cuerpo
columnas_tabla_var = [
    "Fecha",
    "Solidario / TCV Blue",
    "TCV MEP / TCV Blue", 
    "TCV Euro / TCC Blue %",
    "Variación Solidario",
    "Variación TCV Blue",
    "Variación TCV Euro"
]

# Valores correspondientes a los últimos días, 14 columnas
# Definimos una función de "pintado" para las variaciones
def aplicar_color_variacion(valor):
    try:
        val_num = float(valor)
        color = "#27ae60" if val_num > 0 else "#c0392b"
        # El formato .2% convierte 0.012 en 1.20%
        return f'<span style="color: {color}; font-weight: bold;">{val_num:.2%}</span>'
    except:
        return valor

# Mapeamos los datos de la tabla de variaciones
# Filtramos las columnas: Fecha (0) y las variaciones (14 en adelante)
datos_crudos = df[columnas_tabla_var].iloc[:cotizaciones_a_mostrar].values.tolist()

datos_formateados = []
for fila in datos_crudos:
    # fila[0] es la Fecha, la dejamos igual
    # fila[1], [2], [3] son brechas (podés dejarlas igual o pintarlas)
    # fila[4], [5], [6] son Variación_Solidario, Blue y Euro -> ESTAS LAS PINTAMOS
    
    fila_nueva = list(fila)
    # Aplicamos color solo a las últimas 3 columnas de variación
    fila_nueva[-3] = aplicar_color_variacion(fila[-3]) # Solidario
    fila_nueva[-2] = aplicar_color_variacion(fila[-2]) # Blue
    fila_nueva[-1] = aplicar_color_variacion(fila[-1]) # Euro

    datos_formateados.append(fila_nueva)
    
tabla_variaciones = tabulate(
    datos_formateados, 
    headers=columnas_tabla_var,
    tablefmt="unsafehtml",
    numalign="center",
    floatfmt=".2%"
)

# Encabezados para tabular últimos 10 días del df y ponerlo en el cuerpo
inflaciones = [
    "Fecha",
    "I. Mensual",
    "I. Bimestral",
    "I. Trimestral"
]
# Valores correspondientes a los últimos días
valores = inflacion_df[["Fecha", "Inflación Mensual", "Inflación Bimestral", "Inflación Trimestral"]].values.tolist()
tabla_inflacion = tabulate(
    valores,
    headers=inflaciones,
    tablefmt="html",
    numalign="center",
    floatfmt=".2%"
)

# Cálculo de si al comprar dólar oficial se ahorra respecto el blue
cantidad_usd = 100
ahorro_valor = float(df["TCV_Blue"].iloc[0] * cantidad_usd - df["Solidario"].iloc[0] * cantidad_usd)
color_ahorro = "#27ae60" if ahorro_valor > 0 else "#c0392b"
label_ahorro = "Ahorro estimado" if ahorro_valor > 0 else "Sobrecosto estimado"

# --- LÓGICA DE CÁLCULO FUERA DEL HTML ---
costo_blue = float(df["TCV_Blue"].iloc[0] * cantidad_usd)
costo_oficial = float(df["Solidario"].iloc[0] * cantidad_usd)
ahorro_valor = costo_blue - costo_oficial

# Lógica de color dinámica
color_ahorro = "#27ae60" if ahorro_valor > 0 else "#c0392b"
label_ahorro = "Ahorro estimado" if ahorro_valor > 0 else "Sobrecosto estimado"

# Variaciones y Brechas
var_blue = float(((df["TCV_Blue"].iloc[0] / df["TCV_Blue"].iloc[1] - 1) * 100))
brecha_solidario = float((1 - df["Solidario"].iloc[0] / df["TCV_Blue"].iloc[0]) * 100)
var_divisas = float(((df["TCV_Divisas"].iloc[0] / df["TCV_Divisas"].iloc[1] - 1) * 100))
var_euro = float(((df["TCV_Euro"].iloc[0] / df["TCV_Euro"].iloc[1] - 1) * 100))
brecha_euro_blue = (1 - df["TCV_Blue"].iloc[0] / df["TCV_Euro"].iloc[0]) * 100

# Riesgo
riesgo_pts = float(df["riesgo_pais"].iloc[0])
sobretasa = riesgo_pts / 100

# Cuerpo del texto con código html, armado desde templates/report_email.html
jinja_env = Environment(loader=FileSystemLoader("templates"))
template_reporte = jinja_env.get_template("report_email.html")

html = template_reporte.render(
    cantidad_usd=cantidad_usd,
    costo_blue=costo_blue,
    costo_oficial=costo_oficial,
    label_ahorro=label_ahorro,
    color_ahorro=color_ahorro,
    ahorro_valor=ahorro_valor,
    var_blue=var_blue,
    brecha_solidario=brecha_solidario,
    var_divisas=var_divisas,
    var_euro=var_euro,
    brecha_euro_blue=brecha_euro_blue,
    fwd_oficial=fwd_oficial,
    fwd_blue=fwd_blue,
    riesgo_pts=riesgo_pts,
    sobretasa=sobretasa,
    parrafo_ia=parrafo_ia,
    cotizaciones_a_mostrar=cotizaciones_a_mostrar,
    tabla_cotizaciones=tabla_cotizaciones,
    tabla_variaciones=tabla_variaciones,
    tabla_inflacion=tabla_inflacion,
    inflacion_interanual=inflacion_interanual,
    web_bna=web_bna,
    web_dolarhoy=web_dolarhoy,
    web_mep=web_mep,
    api_riesgo_pais=api_riesgo_pais,
    web_euro=web_euro,
    fed_api_url=fed_api_url,
    bcra_api_url=bcra_api_url,
    performance_segundos=time.perf_counter() - comienzo,
)

# Leemos las imágenes una sola vez en el hilo principal: evita que los dos hilos
# de envío abran el mismo archivo al mismo tiempo (en OneDrive esto puede disparar
# un PermissionError transitorio si el archivo está siendo sincronizado)
archivos_img = [graficos_jpg, inflacion_jpg, variaciones_jpg, btc_jpg]
imagenes_bytes = {}
for archivo in archivos_img:
    try:
        with open("Previews/" + archivo, 'rb') as fp:
            imagenes_bytes[archivo] = fp.read()
    except FileNotFoundError:
        print(f"⚠️ No se encontró la imagen: {archivo}")

# Crear el objeto MIMEMultipart y asignarle los campos
def enviar_reporte(receptores, adjuntar_csv=False):
    em = MIMEMultipart('related')
    em['From'] = EMAIL_SENDER
    
    # IMPORTANTE: Si es una lista, usamos BCC (Cco) para privacidad
    # El 'To' lo dejamos como un string genérico o el remitente
    em['To'] = EMAIL_SENDER 
    if isinstance(receptores, list):
        em['Bcc'] = ", ".join(receptores)
        lista_final = receptores # smtplib necesita la lista técnica
    else:
        em['Bcc'] = receptores
        lista_final = [receptores] # convertimos a lista para sendmail
    em['Subject'] = f"📈 Reporte Macroeconómico - {dt.datetime.today():%d-%m-%Y}"
    em.attach(MIMEText(html, 'html'))

    # Adjuntar Imágenes (CIDs), ya leídas en memoria antes de lanzar los hilos
    for i, archivo in enumerate(archivos_img):
        data = imagenes_bytes.get(archivo)
        if data is None:
            continue
        msgImage = MIMEImage(data)
        msgImage.add_header('Content-ID', f'<image{i + 1}>')
        msgImage.add_header('Content-Disposition', 'inline', filename=archivo)
        em.attach(msgImage)

    # Lógica del CSV
    if adjuntar_csv:
        try:
            with open(RUTA_BBDD, 'r', encoding='latin-1') as fp:
                bd_data = fp.read()
            bd = MIMEText(bd_data, 'csv')
            bd.add_header('Content-Disposition', 'attachment', filename='Seguimiento Macroeconómico')
            em.attach(bd)
        except Exception as e:
            print(f"⚠️ Error al adjuntar el CSV: {e}")

    # Envío
    try:
        context = ssl.create_default_context()
        with smtplib.SMTP("smtp.gmail.com", 587) as smtp:
            smtp.ehlo()
            smtp.starttls(context=context)
            smtp.ehlo()
            smtp.login(EMAIL_SENDER, EMAIL_PASSWORD)
            
            # smtplib.sendmail necesita: (Remitente, Lista de todos los destinatarios, Mensaje)
            # Incluimos EMAIL_SENDER en la lista de envío para que el To funcione
            destinatarios_tecnicos = lista_final + [EMAIL_SENDER]
            smtp.sendmail(EMAIL_SENDER, destinatarios_tecnicos, em.as_string())
            
        print(f"🚀 Reporte {'CON CSV' if adjuntar_csv else 'SIN CSV'} enviado con éxito.")
    except Exception as e:
        print(f"❌ Error de red/envío: {e}")

# Enviamos en hilo con y sin CSV para no bloquear el proceso principal
# También se puede enviar secuencialmente, pero vamos a tardar casi el doble
hilo_sin_csv = threading.Thread(
    target=enviar_reporte,
    args=(EMAIL_RECEIVER,),
    kwargs={"adjuntar_csv": False}
)

hilo_con_csv = threading.Thread(
    target=enviar_reporte,
    args=(EMAIL_RECEIVER_CSV,),
    kwargs={"adjuntar_csv": True}
)

hilo_sin_csv.start()  # arranca el primero
hilo_con_csv.start()  # arranca el segundo SIN esperar al primero

hilo_sin_csv.join()   # espera que terminen ambos
hilo_con_csv.join()   # antes de seguir con el resto del script

## 📸 Actualización de Previews en GitHub

In [ ]:
os.chdir(RUTA_REPO)

# Verificar si hubo cambios en /Previews
resultado = subprocess.run(
    ["git", "status", "--porcelain", "Previews"],
    capture_output=True,
    text=True
)

# Si hubo cambios
if resultado.stdout.strip():

    # Agregar solo /Previews
    subprocess.run(["git", "add", "Previews"])

    # Commit
    subprocess.run([
        "git",
        "commit",
        "-m",
        "Automatic previews update"
    ])

    subprocess.run(["git", "push"])

    print("Previews actualizado en GitHub")

else:
    print("No hubo cambios en Previews")

## ⏱ Tiempo de ejecución

In [ ]:
print(f"La duración de la ejecución total fue de: {time.perf_counter() - comienzo:.2f} segundos")